## Dream 2000 Earbuds Scrapping

In [1]:
import selenium 

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
import time

## -- Options --
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")


## -- Earbuds List --
Earbuds = []


## -- Scraping --
driver = webdriver.Chrome(options=options)
driver.get("https://dream2000.com/en/collections/audio")
time.sleep(5)

page = 1

while True:

    page_products = 0

    ## -- Product Cards --
    cards = driver.find_elements(By.CSS_SELECTOR,'.product-card__wrapper.color-scheme-theme.anm-reveal-item.overflow-hidden')

    for card in cards:

        try:
            Product = card.find_element(By.CSS_SELECTOR, 'a[href*="/en/products/"]')

            Name = Product.get_attribute('aria-label')
            Link = Product.get_attribute('href')

            Image = card.find_element(By.CSS_SELECTOR, 'img.product-card__image').get_attribute('src')

        except NoSuchElementException:
            continue

        ## -- Price --
        try:
            Price = card.find_element(By.CSS_SELECTOR, '.price').text

        except NoSuchElementException:
            Price = "Product Not Available"

        ## -- Stock --
        try:
            Stock = card.find_element(By.CSS_SELECTOR,'[data-inventory-status]').get_attribute('data-inventory-status')

        except NoSuchElementException:
            Stock = "Product Not Available"

        Earbuds.append({
            'Name': Name,
            'Link': Link,
            'Image': Image,
            'Price': Price,
            'Stock': Stock
        })

        page_products += 1

    ## -- Page Summary --

    print(
        f"Page {page} finished successfully. "
        f"Products collected on this page: {page_products}. "
        f"Total products collected: {len(Earbuds)}"
    )

    ## -- Next Page --
    next_buttons = driver.find_elements(
        By.CSS_SELECTOR, "a[aria-label='Next']"
    )

    if not next_buttons:
        print("No Next page. Scraping finished.")
        break

    next_url = next_buttons[0].get_attribute('href')

    if not next_url or next_url == driver.current_url:
        print("Invalid or unchanged Next URL.")
        break

    print("Navigating to:", next_url)

    driver.get(next_url)

    page += 1
    time.sleep(5)


## -- Final Result --
print(f"\nScraping completed. Total products: {len(Earbuds)}")


In [ ]:
Earbuds

## Import To CSV & Parquet File

In [4]:
import pandas as pd 

In [6]:
df_Earbuds = pd.DataFrame(Earbuds)

In [7]:
df_Earbuds.head()

,Name,Link,Image,Price,Stock
0,Soundcore by Anker R50I NC A3959H11,https://dream2000.com/en/products/soundcore-by...,https://dream2000.com/cdn/shop/files/anker_sou...,"LE 1,850.00\nLE 1,949.00",in-stock
1,SoundCore by Anker k20i Wireless Earbuds,https://dream2000.com/en/products/anker-k20i-e...,https://dream2000.com/cdn/shop/files/anker-k20...,LE 790.00\nLE 849.00,low-stock
2,SoundCore by Anker R50i VI Earbuds A3969D11,https://dream2000.com/en/products/soundcore-by...,https://dream2000.com/cdn/shop/files/2025-02-0...,"LE 1,120.00\nLE 1,199.00",in-stock
3,Qcy T15 Wireless Earbuds,https://dream2000.com/en/products/qcy-t15-wire...,https://dream2000.com/cdn/shop/files/2026-02-0...,"LE 1,630.00\nLE 1,800.00",in-stock
4,Soundcore by Anker Earbuds 50 Hours ANC R60i N...,https://dream2000.com/en/products/soundcore-ea...,https://dream2000.com/cdn/shop/files/anker_sou...,"LE 2,699.00",in-stock


In [8]:
df_Earbuds.to_csv("Earbuds.csv" , index = False)
df_Earbuds.to_parquet("Earbuds.parquet" , index = False)